# Pytorch進階技巧

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/taipeitechmmslab/MMSLAB-Pytorch/blob/master/Lab6.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/taipeitechmmslab/MMSLAB-Pytorch/blob/master/Lab6.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

In [ ]:
import torch
import torch.nn as nn

## Custom Layers


Example: 簡單客製化的網路計算區塊(Computational Block)設計

通道重排模組

In [ ]:
class ChannelShuffle(nn.Module):
    """ 透過 ChannelShuffle 來進行通道順序的重排, 促進 grouped convolution 的 cross-group 資訊交換 """
    def __init__(self, c, g):
        super(ChannelShuffle, self).__init__()
        self.c = c
        self.g = g

    def forward(self, x):
        batch_size, channels, height, width = x.size()
        channels_per_group = self.c // self.g
        # 將 x reshape 成 [B, g, c // g, H, W]
        x = x.view(batch_size, self.g, channels_per_group, height, width)
        # 透過 transpose 來將 x reshape 成 [B, c // g, g, H, W], 並透過 contiguous 來讓資料在記憶體中連續分布
        x = x.transpose(1, 2).contiguous()
        # 再將 x reshape 成 [B, c, H, W], 這樣就完成了 channel shuffle
        x = x.view(batch_size, -1, height, width)
        return x


網路層架構

In [ ]:
class CustomBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_layers=4, groups=8, act=nn.SiLU):
        """
        此類別為一個自定義的卷積模組, 使用到 nn.ModuleList 和 nn.Sequential 來進行模組的組合
        :param in_channels: 輸入通道數
        :param out_channels: 輸出通道數
        :param num_layers: 中繼層的深度
        :param groups: 進行 group convolution 時的 group 數量, 也就是 cardinality
        :param act: 要使用的 activation function
        """
        super(CustomBlock, self).__init__()
        # 設定中繼層的通道數, 這邊設定成輸出通道數的一半
        self.inter_c = out_channels // 2
        # 設定 skip connection 的連接層
        self.skip_path = nn.Sequential(
            nn.Conv2d(in_channels, self.inter_c, kernel_size=1, groups=groups),
            nn.BatchNorm2d(self.inter_c),
            act(),
        )
        # 設定第一層的卷積層, 主要是進行通道數轉換
        self.first_layer = nn.Sequential(
            nn.Conv2d(in_channels, self.inter_c, kernel_size=1, groups=groups),
            nn.BatchNorm2d(self.inter_c),
            act(),
        )
        # 透過 num_layers 來設定中繼層的數量, 每個中繼層會有兩個 3x3 的卷積來做特徵萃取
        self.layers = nn.ModuleList([
            nn.Sequential(
                # 第一個 3x3 卷積使用 group=inter_c // 4, 並將輸出通道數設定為 inter_c // 2
                nn.Conv2d(self.inter_c, self.inter_c // 2, kernel_size=3, padding=1, groups= self.inter_c // 4),
                nn.BatchNorm2d(self.inter_c // 2),
                act(),
                # 透過 ChannelShuffle 來進行通道順序的重排, 促進 cross-channel 的資訊交換
                ChannelShuffle(self.inter_c // 2, self.inter_c // 4),
                # 第二個 3x3 卷積使用 group=inter_c // 8, 並將輸出通道數設定為 inter_c
                nn.Conv2d(self.inter_c // 2, self.inter_c, kernel_size=3, padding=1, groups=self.inter_c // 8),
                nn.BatchNorm2d(self.inter_c),
                act(),
            ) for _ in range(num_layers)
        ])
        # merge_1x1 會將所有中繼層的輸出通道數進行串接, 並將通道數轉換為輸出通道數
        self.merge_1x1 = nn.Sequential(
            nn.Conv2d(self.inter_c * (num_layers + 1), out_channels, kernel_size=1, groups=groups),
            nn.BatchNorm2d(out_channels),
            act(),
        )
        # residual 若輸入通道數與輸出通道數不同, 則會使用 1x1 的卷積來轉換通道數
        self.residual = nn.Conv2d(in_channels, out_channels, kernel_size=1, groups=groups) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        # residual 分支
        r = self.residual(x)
        # skip 分支
        xs = [self.skip_path(x)]
        # 總共會有 num_layers 個中繼層, 透過 for loop 來進行中繼層的運算
        x = self.first_layer(x)
        for layer in self.layers:
            x = layer(x)
            xs.append(x)
        # 將所有中繼層的輸出和 skip 分支的輸出串接起來
        x = torch.cat(xs, dim=1)
        # 將串接後的結果進行 1x1 卷積, 並將通道數轉換為輸出通道數
        x = self.merge_1x1(x)
        x = x + r
        return x


## Custom Network


In [ ]:
class CustomBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_layers=4, groups=8, act=nn.SiLU):
        """
        此類別為一個自定義的卷積模組, 使用到 nn.ModuleList 和 nn.Sequential 來進行模組的組合
        :param in_channels: 輸入通道數
        :param out_channels: 輸出通道數
        :param num_layers: 中繼層的深度
        :param groups: 進行 group convolution 時的 group 數量, 也就是 cardinality
        :param act: 要使用的 activation function
        """
        super(CustomBlock, self).__init__()
        # 設定中繼層的通道數, 這邊設定成輸出通道數的一半
        self.inter_c = out_channels // 2
        # 設定 skip connection 的連接層
        self.skip_path = nn.Sequential(
            nn.Conv2d(in_channels, self.inter_c, kernel_size=1, groups=groups),
            nn.BatchNorm2d(self.inter_c),
            act(),
        )
        # 設定第一層的卷積層, 主要是進行通道數轉換
        self.first_layer = nn.Sequential(
            nn.Conv2d(in_channels, self.inter_c, kernel_size=1, groups=groups),
            nn.BatchNorm2d(self.inter_c),
            act(),
        )
        # 透過 num_layers 來設定中繼層的數量, 每個中繼層會有兩個 3x3 的卷積來做特徵萃取
        self.layers = nn.ModuleList([
            nn.Sequential(
                # 第一個 3x3 卷積使用 group=inter_c // 4, 並將輸出通道數設定為 inter_c // 2
                nn.Conv2d(self.inter_c, self.inter_c // 2, kernel_size=3, padding=1, groups= self.inter_c // 4),
                nn.BatchNorm2d(self.inter_c // 2),
                act(),
                # 透過 ChannelShuffle 來進行通道順序的重排, 促進 cross-channel 的資訊交換
                ChannelShuffle(self.inter_c // 2, self.inter_c // 4),
                # 第二個 3x3 卷積使用 group=inter_c // 8, 並將輸出通道數設定為 inter_c
                nn.Conv2d(self.inter_c // 2, self.inter_c, kernel_size=3, padding=1, groups=self.inter_c // 8),
                nn.BatchNorm2d(self.inter_c),
                act(),
            ) for _ in range(num_layers)
        ])
        # merge_1x1 會將所有中繼層的輸出通道數進行串接, 並將通道數轉換為輸出通道數
        self.merge_1x1 = nn.Sequential(
            nn.Conv2d(self.inter_c * (num_layers + 1), out_channels, kernel_size=1, groups=groups),
            nn.BatchNorm2d(out_channels),
            act(),
        )
        # residual 若輸入通道數與輸出通道數不同, 則會使用 1x1 的卷積來轉換通道數
        self.residual = nn.Conv2d(in_channels, out_channels, kernel_size=1, groups=groups) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        # residual 分支
        r = self.residual(x)
        # skip 分支
        xs = [self.skip_path(x)]
        # 總共會有 num_layers 個中繼層, 透過 for loop 來進行中繼層的運算
        x = self.first_layer(x)
        for layer in self.layers:
            x = layer(x)
            xs.append(x)
        # 將所有中繼層的輸出和 skip 分支的輸出串接起來
        x = torch.cat(xs, dim=1)
        # 將串接後的結果進行 1x1 卷積, 並將通道數轉換為輸出通道數
        x = self.merge_1x1(x)
        x = x + r
        return x


## Model Wrapper


In [ ]:
class ModelWrapper(object):
    def __init__(
            self,
            model: nn.Module,  # 類神經網路模型
            device: torch.device,  # 使用的裝置
            lr: float = 1e-3,  # learning rate
            optimizer: torch.optim.Optimizer = torch.optim.AdamW,  # 優化器
            criterion: nn.Module = nn.CrossEntropyLoss,  # 損失函數
            metric: callable = None,  # 評估指標
            checkpoint_path: str = None,  # 模型權重的儲存路徑
            load_model: bool = False,  # 是否載入模型權重
            load_optimizer: bool = False,  # 是否載入優化器權重
    ):
        # 初始化相關參數
        self.model = model
        # 實例化優化器
        self.optimizer = optimizer(model.parameters(), lr=lr, weight_decay=5e-3)
        if load_model:
            # 透過自訂的方法 self.load_checkpoint 來載入模型權重與優化器
            self.load_checkpoint(load_optimizer=load_optimizer)
        self.device = device
        # 實例化損失函數與評估指標
        self.criterion = criterion()
        self.metric = metric if metric is not None else self.classification_accuracy
        self.checkpoint_path = checkpoint_path
        # 初始化最佳損失函數, 並設定為無限大
        self.best_loss = float('inf')

    @staticmethod
    def classification_accuracy(logits, y):
        """ 計算分類準確率 """
        pred = torch.argmax(logits, dim=1)
        return (pred == y).float().mean() * 100

    def forward(self, x):
        """ 透過 self.model 對輸入 x 進行向前傳遞, 並回傳結果 """
        return self.model(x)

    def save_checkpoint(self):
        """ 將模型權重與優化器權重儲存至 self.checkpoint_path """
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
        }, self.checkpoint_path)

    def load_checkpoint(self, load_optimizer=True):
        """ 從 self.checkpoint_path 載入模型權重與優化器權重 """
        import os
        assert self.checkpoint_path is not None and os.path.exists(self.checkpoint_path), \
            'checkpoint_path 不可為 None 且路徑必須存在'
        checkpoint = torch.load(self.checkpoint_path)
        # 透過 self.model.load_state_dict 來載入模型權重
        self.model.load_state_dict(checkpoint['model_state_dict'])
        if load_optimizer:
            # 透過 self.optimizer.load_state_dict 來載入優化器權重
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    def get_loss(self, x, y, is_logit=True):
        """ 若輸入的 x 為 logits, 則直接使用 self.criterion 計算 loss, 否則則先將 x 進行向前傳遞, 再計算 loss """
        logit = x if is_logit else self.forward(x)
        return self.criterion(logit, y), logit

    def get_accuracy(self, x, y, is_logit=True):
        """ 若輸入的 x 為 logits, 則直接使用 self.metric 計算 accuracy, 否則則先將 x 進行向前傳遞, 再計算 accuracy """
        logit = x if is_logit else self.forward(x)
        return self.metric(logit, y)

    def record_parameters(self, writer, epoch):
        """ 將模型的參數與梯度紀錄至 tensorboard """
        for name, param in self.model.named_parameters():
            writer.add_histogram(name, param, epoch)

    def train_epoch(self, data_loader, writer=None, epoch=None):
        """ 訓練一個 epoch, 並回傳該 epoch 的平均 loss 與平均 accuracy """
        train_loss, train_accu = 0, 0
        self.model.train()
        for batch in data_loader:
            self.optimizer.zero_grad()
            x, y = batch
            x = x.to(self.device)
            y = y.to(self.device)
            loss, logit = self.get_loss(x, y, is_logit=False)
            loss.backward()
            self.optimizer.step()

            train_loss += loss.item()
            train_accu += self.get_accuracy(logit, y).item()

        train_loss /= len(data_loader)
        train_accu /= len(data_loader)
        if writer is not None:
            writer.add_scalar('loss/train', train_loss, epoch)
            writer.add_scalar('accu/train', train_accu, epoch)
        return train_loss, train_accu

    def eval(self, data_loader, writer=None, epoch=None):
        """ 評估一個 epoch, 並回傳該 epoch 的平均 loss 與平均 accuracy """
        valid_loss, valid_accu = 0, 0
        self.model.eval()
        with torch.no_grad():
            for batch in data_loader:
                x, y = batch
                x = x.to(self.device)
                y = y.to(self.device)
                loss, logit = self.get_loss(x, y, is_logit=False)
                valid_loss += loss.item()
                valid_accu += self.get_accuracy(logit, y).item()

        valid_loss /= len(data_loader)
        valid_accu /= len(data_loader)
        if writer is not None:
            writer.add_scalar('loss/valid', valid_loss, epoch)
            writer.add_scalar('accu/valid', valid_accu, epoch)

        if valid_loss < self.best_loss:
            # 若驗證集的 loss 優於目前最佳的 loss, 則更新 self.best_loss, 並儲存模型權重
            self.best_loss = valid_loss
            self.save_checkpoint()

        return valid_loss, valid_accu


# 實驗：透過Model Wrapper來進行兩種不同的網路架構訓練

### Import 必要套件

In [ ]:
!pip install torchinfo

import os
import shutil
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms as T
import torchvision
from torchinfo import summary
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

# 初始化計算裝置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### 定義工具類別與資料增強函式

包含訓練邏輯的 ModelWrapper 以及影像前處理的 image_transform_loader

In [ ]:
class ModelWrapper(object):
    def __init__(
            self,
            model: nn.Module,  # 類神經網路模型
            device: torch.device,  # 使用的裝置
            lr: float = 1e-3,  # learning rate
            optimizer: torch.optim.Optimizer = torch.optim.AdamW,  # 優化器
            criterion: nn.Module = nn.CrossEntropyLoss,  # 損失函數
            metric: callable = None,  # 評估指標
            checkpoint_path: str = None,  # 模型權重的儲存路徑
            load_model: bool = False,  # 是否載入模型權重
            load_optimizer: bool = False,  # 是否載入優化器權重
    ):
        self.model = model
        self.optimizer = optimizer(model.parameters(), lr=lr, weight_decay=5e-3)
        self.checkpoint_path = checkpoint_path
        
        if load_model:
            self.load_checkpoint(load_optimizer=load_optimizer)
            
        self.device = device
        self.criterion = criterion()
        self.metric = metric if metric is not None else self.classification_accuracy
        self.best_loss = float('inf')

    @staticmethod
    def classification_accuracy(logits, y):
        """ 計算分類準確率 """
        pred = torch.argmax(logits, dim=1)
        return (pred == y).float().mean() * 100

    def forward(self, x):
        """ 透過 self.model 對輸入 x 進行向前傳遞, 並回傳結果 """
        return self.model(x)

    def save_checkpoint(self):
        """ 將模型權重與優化器權重儲存 """
        if self.checkpoint_path is None:
            return
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
        }, self.checkpoint_path)

    def load_checkpoint(self, load_optimizer=True):
        """ 從 checkpoint_path 載入模型權重與優化器權重 """
        assert self.checkpoint_path is not None and os.path.exists(self.checkpoint_path), \
            'checkpoint_path 不可為 None 且路徑必須存在'
        checkpoint = torch.load(self.checkpoint_path)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        if load_optimizer:
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    def get_loss(self, x, y, is_logit=True):
        logit = x if is_logit else self.forward(x)
        return self.criterion(logit, y), logit

    def get_accuracy(self, x, y, is_logit=True):
        logit = x if is_logit else self.forward(x)
        return self.metric(logit, y)

    def record_parameters(self, writer, epoch):
        """ 將模型的參數與梯度紀錄至 tensorboard """
        for name, param in self.model.named_parameters():
            writer.add_histogram(name, param, epoch)

    def train_epoch(self, data_loader, writer=None, epoch=None):
        train_loss, train_accu = 0, 0
        self.model.train()
        for batch in data_loader:
            self.optimizer.zero_grad()
            x, y = batch
            x, y = x.to(self.device), y.to(self.device)
            
            loss, logit = self.get_loss(x, y, is_logit=False)
            loss.backward()
            self.optimizer.step()

            train_loss += loss.item()
            train_accu += self.get_accuracy(logit, y).item()

        train_loss /= len(data_loader)
        train_accu /= len(data_loader)
        
        if writer is not None:
            writer.add_scalar('loss/train', train_loss, epoch)
            writer.add_scalar('accu/train', train_accu, epoch)
            
        return train_loss, train_accu

    def eval(self, data_loader, writer=None, epoch=None):
        valid_loss, valid_accu = 0, 0
        self.model.eval()
        with torch.no_grad():
            for batch in data_loader:
                x, y = batch
                x, y = x.to(self.device), y.to(self.device)
                
                loss, logit = self.get_loss(x, y, is_logit=False)
                valid_loss += loss.item()
                valid_accu += self.get_accuracy(logit, y).item()

        valid_loss /= len(data_loader)
        valid_accu /= len(data_loader)
        
        if writer is not None:
            writer.add_scalar('loss/valid', valid_loss, epoch)
            writer.add_scalar('accu/valid', valid_accu, epoch)

        if valid_loss < self.best_loss:
            self.best_loss = valid_loss
            self.save_checkpoint()

        return valid_loss, valid_accu


def image_transform_loader(img_size, with_aug=False, p=.5, flip_h=False, flip_v=False,
                           color=False, contrast=False, sharpness=False, crop_rand=False,
                           crop_center=False, blur=False, rotate=False):
    transform_list = [T.ToTensor()]
    if with_aug:
        if flip_h:
            transform_list += [T.RandomHorizontalFlip(p)]
        if flip_v:
            transform_list += [T.RandomVerticalFlip(p)]
        if color:
            transform_list += [T.ColorJitter(brightness=.5, hue=.3)]
        if contrast:
            transform_list += [T.RandomAutocontrast(p)]
        if sharpness:
            transform_list += [T.RandomAdjustSharpness(sharpness_factor=2, p=p)]
        if blur:
            transform_list += [T.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5))]
        if crop_rand:
            to_size = int(img_size * .8)
            transform_list += [T.RandomCrop(size=(to_size, to_size))]
        if crop_center:
            transform_list += [T.CenterCrop(size=img_size)]
        if rotate:
            transform_list += [T.RandomRotation(degrees=5)]
            
    # PyTorch 新版本建議使用 antialias=True
    transform_list += [T.Resize(size=img_size, antialias=True)]
    transform_list += [T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
    return T.Compose(transform_list)

### 定義網路架構

In [ ]:
class ChannelShuffle(nn.Module):
    """ 透過 ChannelShuffle 來進行通道順序的重排, 促進 grouped convolution 的 cross-group 資訊交換 """
    def __init__(self, c, g):
        super(ChannelShuffle, self).__init__()
        self.c = c
        self.g = g

    def forward(self, x):
        batch_size, channels, height, width = x.size()
        channels_per_group = self.c // self.g
        x = x.view(batch_size, self.g, channels_per_group, height, width)
        x = x.transpose(1, 2).contiguous()
        x = x.view(batch_size, -1, height, width)
        return x


class CustomBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_layers=4, groups=8, act=nn.SiLU):
        super(CustomBlock, self).__init__()
        self.inter_c = out_channels // 2
        
        self.skip_path = nn.Sequential(
            nn.Conv2d(in_channels, self.inter_c, kernel_size=1, groups=groups),
            nn.BatchNorm2d(self.inter_c),
            act(),
        )
        
        self.first_layer = nn.Sequential(
            nn.Conv2d(in_channels, self.inter_c, kernel_size=1, groups=groups),
            nn.BatchNorm2d(self.inter_c),
            act(),
        )
        
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(self.inter_c, self.inter_c // 2, kernel_size=3, padding=1, groups=self.inter_c // 4),
                nn.BatchNorm2d(self.inter_c // 2),
                act(),
                ChannelShuffle(self.inter_c // 2, self.inter_c // 4),
                nn.Conv2d(self.inter_c // 2, self.inter_c, kernel_size=3, padding=1, groups=self.inter_c // 8),
                nn.BatchNorm2d(self.inter_c),
                act(),
            ) for _ in range(num_layers)
        ])
        
        self.merge_1x1 = nn.Sequential(
            nn.Conv2d(self.inter_c * (num_layers + 1), out_channels, kernel_size=1, groups=groups),
            nn.BatchNorm2d(out_channels),
            act(),
        )
        
        self.residual = nn.Conv2d(in_channels, out_channels, kernel_size=1, groups=groups) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        r = self.residual(x)
        xs = [self.skip_path(x)]
        x = self.first_layer(x)
        for layer in self.layers:
            x = layer(x)
            xs.append(x)
            
        x = torch.cat(xs, dim=1)
        x = self.merge_1x1(x)
        x = x + r
        return x


class CustomModel(nn.Module):
    def __init__(self, num_class, num_blk=None, num_layer=3, num_group=2):
        super(CustomModel, self).__init__()
        if num_blk is None:
            num_blk = [1, 2, 2]

        base_d = 16 * num_layer
        act = nn.SiLU
        self.pre_conv = nn.Sequential(
            nn.Conv2d(3, base_d, kernel_size=7),
            nn.BatchNorm2d(base_d),
            act(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        
        self.convolution_blocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(base_d, base_d * 2, kernel_size=1),
                nn.BatchNorm2d(base_d * 2),
                act(),
            ),
            nn.Sequential(*[
                CustomBlock(base_d * 2, base_d * 2, num_layers=num_layer, groups=num_group) for _ in range(num_blk[0])
            ]),
            nn.Sequential(
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
                nn.Conv2d(base_d * 2, base_d * 4, kernel_size=1),
                nn.BatchNorm2d(base_d * 4),
                act(),
            ),
            nn.Sequential(*[
                CustomBlock(base_d * 4, base_d * 4, num_layers=num_layer, groups=num_group) for _ in range(num_blk[1])
            ]),
            nn.Sequential(
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
                nn.Conv2d(base_d * 4, base_d * 8, kernel_size=1),
                nn.BatchNorm2d(base_d * 8),
                act(),
            ),
            nn.Sequential(*[
                CustomBlock(base_d * 8, base_d * 8, num_layers=num_layer, groups=num_group) for _ in range(num_blk[2])
            ]),
            nn.Sequential(
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
                nn.Conv2d(base_d * 8, base_d * 2, kernel_size=1),
                nn.BatchNorm2d(base_d * 2),
                act(),
            ),
            nn.Sequential(
                nn.Flatten(),
                nn.Dropout(.2),
                nn.LazyLinear(64),
                act(),
                nn.Dropout(.5),
                nn.LazyLinear(num_class, bias=False),
            ),
        ])

    def forward(self, x):
        x = self.pre_conv(x)
        for layer in self.convolution_blocks:
            x = layer(x)
        return x


class ConvNet(nn.Module):
    """ 卷積神經網路, 可以選擇是否使用 batch normalization """
    def __init__(self, use_bn=False, class_num=10):
        super().__init__()
        place_holder = lambda x: nn.BatchNorm2d(x) if use_bn else nn.Identity()
        place_holder_1d = lambda x: nn.BatchNorm1d(x) if use_bn else nn.Identity()
        
        self.layers = nn.ModuleList([
            nn.Conv2d(3, 64, (3, 3), padding=1),
            place_holder(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 2), (2, 2)),
            nn.Conv2d(64, 128, (3, 3)),
            place_holder(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, (3, 3), padding=1),
            place_holder(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, (3, 3)),
            nn.MaxPool2d((2, 2), (2, 2)),
            place_holder(256),
            nn.ReLU(),
            nn.Conv2d(256, 128, (3, 3)),
            nn.MaxPool2d((2, 2), (2, 2)),
            place_holder(128),
            nn.ReLU(),
            nn.Conv2d(128, 64, (3, 3)),
            place_holder(64),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            place_holder_1d(64),
            nn.ReLU(),
            nn.Dropout(.5),
            nn.Linear(64, class_num, bias=False),
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

### CIFAR10 Dataset 設定
處理影像前處理與資料集的載入

In [ ]:
batch_size = 128
cpu_num = 3 if os.cpu_count() > 3 else os.cpu_count()
chk_os = False
if chk_os and os.name == 'nt':
    cpu_num = 0

img_size = 64
transform = image_transform_loader(img_size)
transform_aug = image_transform_loader(
    img_size, with_aug=True, rotate=True, flip_h=True, contrast=True, sharpness=True
)

target_dataset = 1  # 0: CIFAR100, 1: CIFAR10

if target_dataset == 0:
    dataset_name = 'CIFAR100'
    dataset = torchvision.datasets.CIFAR100(root='./data', train=True, transform=transform, download=True)
    dataset_aug = torchvision.datasets.CIFAR100(root='./data', train=True, transform=transform_aug, download=True)
    class_num = 100
else:
    dataset_name = 'CIFAR10'
    dataset = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
    dataset_aug = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform_aug, download=True)
    class_num = 10

d_len = len(dataset)

# 固定隨機數種子進行資料集切割
np.random.seed(9527)
indices = np.arange(d_len)
np.random.shuffle(indices)
np.random.seed()

train_indices = indices[:int(d_len * .7)]
valid_indices = indices[int(d_len * .7):]

train_subset = torch.utils.data.Subset(dataset_aug, train_indices)
valid_subset = torch.utils.data.Subset(dataset, valid_indices)

loader_train = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=cpu_num, pin_memory=True)
loader_valid = DataLoader(valid_subset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

# 抽樣一筆資料，供後續 TensorBoard 視覺化使用
dataiter = iter(loader_train)
images, labels = next(dataiter)
images = images.to(device)

### 執行模型訓練迴圈

In [ ]:
model_dir = os.path.join('models', dataset_name)
weight_dir = os.path.join('weights', dataset_name)

# 清除目前 dataset 舊的紀錄，避免 TensorBoard 比較時混到歷史結果
shutil.rmtree(model_dir, ignore_errors=True)
shutil.rmtree(weight_dir, ignore_errors=True)
os.makedirs(model_dir, exist_ok=True)
os.makedirs(weight_dir, exist_ok=True)

model_custom = CustomModel(class_num).to(device)
model_conv = ConvNet(use_bn=True, class_num=class_num).to(device)

model_seq = zip(
    ('ConvNet', 'CustomModel'),  # 模型名稱
    (model_conv, model_custom)   # 模型
)

for model_name, model in model_seq:
    writer = SummaryWriter(os.path.join(model_dir, model_name))
    
    # 向前傳遞一次, 讓模型的 LazyLayer (如 LazyLinear) 初始化權重維度
    model(images)
    
    # 使用 tensorboard 紀錄模型架構
    model.eval()
    writer.add_graph(model, images)

    epochs = 50
    lr = 1e-3
    running_loss_train, running_accu_train, running_loss_valid, running_accu_valid = 0, 0, 0, 0
    
    checkpoint_path = os.path.join(weight_dir, f'{model_name}.pt')
    model_wrapper = ModelWrapper(model, device, lr, checkpoint_path=checkpoint_path)
    
    for i in range(epochs):
        # 使用 tqdm 顯示 Train 進度
        postfix = {'loss': f"{running_loss_train:.4f}", 'accu': f"{running_accu_train:.2f}%"} if i > 0 else {}
        tqdm_loader_train = tqdm(
            loader_train,
            desc=f'[{dataset_name}] {model_name} Epoch {i + 1}/{epochs} [Train]',
            dynamic_ncols=True,
            postfix=postfix
        )
        
        loss_train, accu_train = model_wrapper.train_epoch(tqdm_loader_train, writer, i)
        running_loss_train = loss_train
        running_accu_train = accu_train

        # 使用 tqdm 顯示 Validation 進度
        postfix = {'loss': f"{running_loss_valid:.4f}", 'accu': f"{running_accu_valid:.2f}%"} if i > 0 else {}
        tqdm_loader_valid = tqdm(
            loader_valid,
            desc=f'[{dataset_name}] {model_name} Epoch {i + 1}/{epochs} [Validation]',
            dynamic_ncols=True,
            postfix=postfix
        )
        
        loss_val, accu_val = model_wrapper.eval(tqdm_loader_valid, writer, i)
        running_loss_valid = loss_val
        running_accu_valid = accu_val

        # 每 5 個 epoch 儲存並紀錄參數
        if i % 5 == 0:
            model_wrapper.save_checkpoint()
            model_wrapper.record_parameters(writer, i)
            
    writer.close()

### CIFAR100 Dataset 設定
處理影像前處理與資料集的載入

In [ ]:
batch_size = 128
cpu_num = 3 if os.cpu_count() > 3 else os.cpu_count()
chk_os = False
if chk_os and os.name == 'nt':
    cpu_num = 0

img_size = 64
transform = image_transform_loader(img_size)
transform_aug = image_transform_loader(
    img_size, with_aug=True, rotate=True, flip_h=True, contrast=True, sharpness=True
)

target_dataset = 0  # 0: CIFAR100, 1: CIFAR10

if target_dataset == 0:
    dataset_name = 'CIFAR100'
    dataset = torchvision.datasets.CIFAR100(root='./data', train=True, transform=transform, download=True)
    dataset_aug = torchvision.datasets.CIFAR100(root='./data', train=True, transform=transform_aug, download=True)
    class_num = 100
else:
    dataset_name = 'CIFAR10'
    dataset = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
    dataset_aug = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform_aug, download=True)
    class_num = 10

d_len = len(dataset)

# 固定隨機數種子進行資料集切割
np.random.seed(9527)
indices = np.arange(d_len)
np.random.shuffle(indices)
np.random.seed()

train_indices = indices[:int(d_len * .7)]
valid_indices = indices[int(d_len * .7):]

train_subset = torch.utils.data.Subset(dataset_aug, train_indices)
valid_subset = torch.utils.data.Subset(dataset, valid_indices)

loader_train = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=cpu_num, pin_memory=True)
loader_valid = DataLoader(valid_subset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

# 抽樣一筆資料，供後續 TensorBoard 視覺化使用
dataiter = iter(loader_train)
images, labels = next(dataiter)
images = images.to(device)

### 執行模型訓練迴圈

In [ ]:
model_dir = os.path.join('models', dataset_name)
weight_dir = os.path.join('weights', dataset_name)

# 清除目前 dataset 舊的紀錄，避免 TensorBoard 比較時混到歷史結果
shutil.rmtree(model_dir, ignore_errors=True)
shutil.rmtree(weight_dir, ignore_errors=True)
os.makedirs(model_dir, exist_ok=True)
os.makedirs(weight_dir, exist_ok=True)

model_custom = CustomModel(class_num).to(device)
model_conv = ConvNet(use_bn=True, class_num=class_num).to(device)

model_seq = zip(
    ('ConvNet', 'CustomModel'),  # 模型名稱
    (model_conv, model_custom)   # 模型
)

for model_name, model in model_seq:
    writer = SummaryWriter(os.path.join(model_dir, model_name))
    
    # 向前傳遞一次, 讓模型的 LazyLayer (如 LazyLinear) 初始化權重維度
    model(images)
    
    # 使用 tensorboard 紀錄模型架構
    model.eval()
    writer.add_graph(model, images)

    epochs = 50
    lr = 1e-3
    running_loss_train, running_accu_train, running_loss_valid, running_accu_valid = 0, 0, 0, 0
    
    checkpoint_path = os.path.join(weight_dir, f'{model_name}.pt')
    model_wrapper = ModelWrapper(model, device, lr, checkpoint_path=checkpoint_path)
    
    for i in range(epochs):
        # 使用 tqdm 顯示 Train 進度
        postfix = {'loss': f"{running_loss_train:.4f}", 'accu': f"{running_accu_train:.2f}%"} if i > 0 else {}
        tqdm_loader_train = tqdm(
            loader_train,
            desc=f'[{dataset_name}] {model_name} Epoch {i + 1}/{epochs} [Train]',
            dynamic_ncols=True,
            postfix=postfix
        )
        
        loss_train, accu_train = model_wrapper.train_epoch(tqdm_loader_train, writer, i)
        running_loss_train = loss_train
        running_accu_train = accu_train

        # 使用 tqdm 顯示 Validation 進度
        postfix = {'loss': f"{running_loss_valid:.4f}", 'accu': f"{running_accu_valid:.2f}%"} if i > 0 else {}
        tqdm_loader_valid = tqdm(
            loader_valid,
            desc=f'[{dataset_name}] {model_name} Epoch {i + 1}/{epochs} [Validation]',
            dynamic_ncols=True,
            postfix=postfix
        )
        
        loss_val, accu_val = model_wrapper.eval(tqdm_loader_valid, writer, i)
        running_loss_valid = loss_val
        running_accu_valid = accu_val

        # 每 5 個 epoch 儲存並紀錄參數
        if i % 5 == 0:
            model_wrapper.save_checkpoint()
            model_wrapper.record_parameters(writer, i)
            
    writer.close()

### 參數量比較

In [ ]:
model_conv = ConvNet(use_bn=True, class_num=class_num).to(device)
print(summary(model_conv, input_size=(1, 3, img_size, img_size)))

In [ ]:

model_custom = CustomModel(class_num).to(device)
print(summary(model_custom, input_size=(1, 3, img_size, img_size)))

### TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir models